In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# load general packages
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import copy

# load modules related to this exercise
from model import model_bufferstock
import estimate

# Exercise 4: Estimating the buffer-stock consumption model with MLE and MSM

This exercise uses the buffer-stock model to estimate preference parameters by maximum likelihood (MLE) and the method of simulated moments (MSM). You will complete the objective-function calculations in `estimate.py`.


## How EGM enters estimation

Both estimators repeatedly use the structural model as an inner loop:

$$
\text{candidate parameters}
\longrightarrow \text{construct grids and solve by EGM}
\longrightarrow \text{predict consumption or moments}
\longrightarrow \text{evaluate the objective}
\longrightarrow \text{optimizer proposes new parameters}.
$$

This is why solution speed matters economically: every likelihood or MSM evaluation may require another model solution. The synthetic data below are generated once from the baseline calibration. During estimation, `estimate.updatepar` changes candidate parameters, `model.create_grids()` rebuilds derived objects, and `model.solve()` recomputes the policies.


## 1. Ensure that you *understand* the following sections and functions:
<ol type="a">
<li> sections a) and b)</li>
<li> estimate.updatepar </li>
<li> estimate.maximum_likelihood </li>
</ol>

### a) Solve

In [ ]:
# settings, solve and simulate
model = model_bufferstock()
model.life_cycle_setup()
model.create_grids()
model.solve()
model.simulate()

### b) Create data set

In [ ]:
par = model.par
sol = model.sol
sim = model.sim

par.sigma_eta = 0.1

class data: pass
data.t = 20 # time period used for estimation
data.M = sim.M[data.t,:]
data.P = sim.P[data.t,:]
data.m = sim.m[data.t,:]

# add noise to simulated data
data.logC = np.log(sim.C[data.t,:]) - np.random.normal(scale = par.sigma_eta, size = (1, par.simN))

## 2. Complete `estimate.log_likelihood`.


## 3. Inspect the likelihood surface and estimate the model by MLE.


### c) Illustrate likelihood function

In [ ]:
# 1. copy the true parameters,
par_beta = copy.copy(par.beta)
par_rho = copy.copy(par.rho)

# 2. make grids for parameters
Nbeta = 20
Nrho = 15
beta = np.linspace(0.9,0.97,Nbeta)
rho = np.linspace(1.1,4,Nrho)

# 3. allocate
log_lik = np.zeros((Nbeta, Nrho)) + np.nan

# 4. find the log-likelihood for each combination of beta and rho
for i in range(Nbeta):

    print(i) # print i, and thereby show how far the code is

    for j in range(Nrho):

        est_par = ['beta','rho']
        theta0 = [beta[i], rho[j]]
        log_lik[i,j]=estimate.log_likelihood(theta0, model, est_par, data)

# 5. re-inset the true parameters 
par.beta = copy.copy(par_beta)
par.rho = copy.copy(par_rho)

In [ ]:
# 1. set up figure
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1,1,1,projection='3d')

# 2. make data
X, Y = np.meshgrid(rho, beta)

# 3. plot the surface.
surf = ax.plot_surface(X, Y, log_lik, cmap=cm.jet)

# 4. customize the axis.
ax.set_xlabel(r'$\rho$')
ax.set_ylabel(r'$\beta$')
ax.set_title(r'Log-likelihood $(\rho, \beta)$')
ax.invert_xaxis()

# 5. add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)

In [ ]:
# index of parameters closest to the true
i_rho = abs(par_rho - rho).argmin(0)
i_beta = abs(par_beta - beta).argmin(0)

fig, ax = plt.subplots(1, 2, figsize=(20,5))

ax[0].plot(beta,log_lik[:,i_rho])
ax[0].set_xlabel(r'$\rho$')
ax[0].set_title(rf'Log-Likelihood given $\rho$ = {rho[i_rho]:.2f} ')

ax[1].plot(rho,log_lik[i_beta,:])
ax[1].set_xlabel(r'$\beta$')
ax[1].set_title(rf'Log-likelihood given $\beta$ = {beta[i_beta]:.4f}');

### d) Estimate by MLE

In [ ]:
est_par = ['beta'] # parameter to estimate
theta0 = [0.94] # initial guess

est = estimate.maximum_likelihood(model, est_par,theta0,data)

# re-inset the true parameters 
par.beta = copy.copy(par_beta)

print(f'Log-Likelihood:          {-est.fun:.4f}')
print(f'beta:                    {est.x[0]:.4f}')
print(f'Number of iterartions:   {est.nit}')

In [ ]:
est_par = ['rho'] # parameter to estimate
theta0 = [4] # initial guess

est = estimate.maximum_likelihood(model, est_par,theta0,data)

# re-inset the true parameters 
par.rho = copy.copy(par_rho)

print(f'Log-Likelihood:          {-est.fun:.4f}')
print(f'rho:                     {est.x[0]:.4f}')
print(f'Number of iterartions:   {est.nit}')

In [ ]:
est_par = ['rho', 'beta'] # parameters to estimate
theta0 = [3, 0.94] # initial guesses

est = estimate.maximum_likelihood(model, est_par, theta0, data)

# re-inset the true parameters 
par.beta = copy.copy(par_beta)
par.rho = copy.copy(par_rho)

print(f'Log-Likelihood:          {-est.fun:.4f}')
print(f'rho:                     {est.x[0]:.4f}')
print(f'beta:                    {est.x[1]:.4f}')
print(f'Number of iterations:    {est.nit}')

## 4. Ensure that you *understand* the following section and functions:
<ol type="a">
<li> section e) </li>
<li> estimate.calc_moments </li>
<li> estimate.method_simulated_moments </li>
</ol>


### e) MSM Settings

In [ ]:
par.simN = 50000
par.moments_minage = 40
par.moments_maxage = 55
par.moments_numsim = 1
data = copy.copy(sim)
data.moments = estimate.calc_moments(par,data)

## 5. Complete `estimate.sum_squared_diff_moments`.


## 6. Inspect the MSM objective surface and estimate the model by MSM.


### f) Illustrate MSM

In [ ]:
# 1. make grids for parameters
Nbeta = 20
Nrho = 15
beta = np.linspace(0.9,0.97,Nbeta)
rho = np.linspace(1.1,4,Nrho)

# 2. allocate
obj = np.zeros((Nbeta,Nrho)) + np.nan

# 3. find objective function for each combination of beta and rho
for i in range(Nbeta):

    print(i) 

    for j in range(Nrho):

        est_par = ['beta','rho']
        theta0 = [beta[i], rho[j]]
        obj[i,j]=estimate.sum_squared_diff_moments(theta0,model,est_par,data)

# 4. re-inset the true parameters 
par.beta = copy.copy(par_beta)
par.rho = copy.copy(par_rho)

In [ ]:
# 1. plot figure in three dimensions
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(1,1,1,projection='3d')

# 2. make data
X, Y = np.meshgrid(rho, beta)

# 3. plot the surface.
surf = ax.plot_surface(X, Y, obj, cmap=cm.jet)

# 4. customize the axis.
ax.set_xlabel(r'$\rho$')
ax.set_ylabel(r'$\beta$')
ax.set_title(rf'Mehod of Simulated Moments $(\rho, \beta)$')
ax.set_xlim(1.0,4.0)
ax.set_ylim(0.9,0.98)
ax.invert_xaxis()

# 5. add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)

In [ ]:
# index of parameters closest to true parameter
i_rho = abs(par_rho - rho).argmin(0)
i_beta = abs(par_beta - beta).argmin(0)

fig, ax = plt.subplots(1, 2, figsize=(20,5))

ax[0].plot(beta, obj[:, i_rho])
ax[0].set_xlabel(r'$\rho$')
ax[0].set_title(rf'Method of simulated moment given $\rho$ = {rho[i_rho]:.2f} ')

ax[1].plot(rho, obj[i_beta, :])
ax[1].set_xlabel(r'$\beta$')
ax[1].set_title(rf'Method of simulated moment given $\beta$ = {beta[i_beta]:.2f}');

### g) Estimate by MSM

In [ ]:
est_par = ['beta'] # parameter to estimate
theta0 = [0.92] # initial guess

est = estimate.method_simulated_moments(model, est_par, theta0, data)

# re-inset the true parameters 
par.beta = copy.copy(par_beta)

print(f'Objective:               {est.fun:.4f}')
print(f'beta:                    {est.x[0]:.4f}')
print(f'Number of iterartions:   {est.nit}')


In [ ]:
est_par = ['rho'] # parameter to estimate
theta0 = [4] # initial guess

est = estimate.method_simulated_moments(model, est_par, theta0, data)

# re-inset the true parameters 
par.rho = copy.copy(par_rho)

print(f'Objective:               {est.fun:.4f}')
print(f'rho:                     {est.x[0]:.4f}')
print(f'Number of iterartions:   {est.nit}')


In [ ]:
est_par = ['rho','beta'] # parameters to estimate
theta0 = [4, 0.92] # initial guesses

est = estimate.method_simulated_moments(model, est_par,theta0,data)

# re-inset the true parameters 
par.beta = copy.copy(par_beta)
par.rho = copy.copy(par_rho)

print(f'Objective:               {est.fun:.4f}')
print(f'rho:                     {est.x[0]:.4f}')
print(f'beta:                    {est.x[1]:.4f}')
print(f'Number of iterartions:   {est.nit}')
